In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import os
current_pwd = os.getcwd()

possible_paths = [
    '/home/export/soheuny/SRFinder/soheun/notebooks', 
    '/home/soheuny/HH4bsim/soheun/notebooks'
]
    
assert os.getcwd() in possible_paths, f"Did you change the path? It should be one of {possible_paths}"
os.chdir("..")

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from plots import hist_events_by_labels
from events_data import EventsData
from fvt_classifier import FvTClassifier
# import LogNorm
from matplotlib.colors import LogNorm
from training_info import TrainingInfo
from plots import plot_rewighted_samples_by_model, plot_samples_raw
from dataset import MotherSamples
from events_data import events_from_scdinfo
import pickle

features = [
    "sym_Jet0_pt", "sym_Jet1_pt", "sym_Jet2_pt", "sym_Jet3_pt",
    "sym_Jet0_eta", "sym_Jet1_eta", "sym_Jet2_eta", "sym_Jet3_eta",
    "sym_Jet0_phi", "sym_Jet1_phi", "sym_Jet2_phi", "sym_Jet3_phi",  
    "sym_Jet0_m", "sym_Jet1_m", "sym_Jet2_m", "sym_Jet3_m",
]

# use tex
plt.rcParams["text.usetex"] = True
# plt.rcParams["font.family"] = "serif"
# plt.rcParams["font.serif"] = "Times New Roman"

plt.rcParams["figure.dpi"] = 100
plt.rcParams["figure.titlesize"] = 20
plt.rcParams["axes.titlesize"] = 20
plt.rcParams["axes.labelsize"] = 15
plt.rcParams["figure.labelsize"] = 20
plt.rcParams["lines.markersize"] = 3

import pandas as pd

path_3b = Path("../events/MG3/dataframes/threeTag_picoAOD.h5")
path_4b = Path("../events/MG3/dataframes/fourTag_10x_picoAOD.h5")
path_signal = Path("../events/MG3/dataframes/HH4b_picoAOD.h5")
df_3b = pd.read_hdf(path_3b)
df_bg4b = pd.read_hdf(path_4b)
df_signal = pd.read_hdf(path_signal)
df_3b["signal"] = False
df_bg4b["signal"] = False
df_signal["signal"] = True
raw_df_list = [df_3b, df_bg4b, df_signal]
loaded_df = {path_3b: df_3b, path_4b: df_bg4b, path_signal: df_signal}

In [3]:
metadata = TrainingInfo.load_metadata()

In [4]:
base_exp_name = "CR_fvt_training_ensemble_max"
pull_dicts = {}

for ensemble_mode, bin_stats in [("max", "smeared"), ("max", "fvt"), ("mean", "smeared"), ("mean", "fvt")]:
    fname = f"pull_by_hashes_{bin_stats}_{ensemble_mode}_{base_exp_name}"
    pull_dicts[(ensemble_mode, bin_stats)] = pickle.load(open(f"./data/pulls/{fname}.pkl", "rb"))

In [5]:
def get_pull(num: np.array, den: np.array):
    pull = np.zeros_like(num)
    nz = den > 0
    pull[nz] = num[nz] / np.sqrt(den[nz])
    return pull

for ensemble_mode, bin_stats in [("max", "smeared"), ("max", "fvt"), ("mean", "smeared"), ("mean", "fvt")]:
    for hash_ in pull_dicts[(ensemble_mode, bin_stats)]:
        hparams = metadata[hash_]
        signal_ratio = hparams["dataset"]["signal_ratio"]
        SR_size = hparams["signal_region"]["4b_in_SR"]    
        sr_stats_hash = hparams["signal_region"]["SR_stats_hashes"][0]
        sr_stats_type = hparams["signal_region"]["stats_type"]
        if sr_stats_type == "fvt":
            noise_scale = np.inf
        else:
            noise_scale = metadata[sr_stats_hash]["smearing"]["noise_scale"]
        
        for v in pull_dicts[(ensemble_mode, bin_stats)][hash_]:
            binning_mode = v["binning_mode"]
            nbins = v["nbins"]
            hists = v["hists"]
            hist_3b_rw = hists["3b_rw"]
            hist_3b_rw_sq = hists["3b_rw_sq"]
            hist_4b = hists["4b"]
            hist_4b_sq = hists["4b_sq"]
            var_o1 = hists["var_o1"]
            var_o2 = hists["var_o2"]
            hist_3b_o1 = hists["3b_corrected_o1"]
            hist_3b_o2 = hists["3b_corrected_o2"]
            hist_signal = hists["signal"]
            hist_bg4b = hists["bg4b"]
            
            if binning_mode == "train":
                var_o0 = hist_4b_sq + hist_3b_rw_sq
            else:
                var_o0 = hist_4b_sq + hist_3b_rw_sq / nbins**2

            v["signal_ratio"] = signal_ratio
            v["SR_size"] = SR_size
            v["noise_scale"] = noise_scale
            v["n_eff_bins"] = np.sum(var_o0 > 0)
            pull_o0 = get_pull(hist_4b - hist_3b_rw, np.sqrt(var_o0))
            pull_o1 = get_pull(hist_4b - hist_3b_o1, np.sqrt(var_o1))
            pull_o2 = get_pull(hist_4b - hist_3b_o2, np.sqrt(var_o2))
            pull_signal = get_pull(hist_signal, np.sqrt(var_o0))
            pull_bg4b = get_pull(hist_bg4b - hist_3b_rw, np.sqrt(var_o0))
            v["ensemble_mode"] = ensemble_mode
            v["bin_stats"] = bin_stats

In [9]:
# only filter out null case
pull_dicts_null = {k: v for k, v in pull_dicts[("max", "smeared")].items() if (v[0]["signal_ratio"] == 0 
                                                                               and v[0]["SR_size"] == 0.15
                                                                               and v[0]["noise_scale"] == 2.0)}    

In [11]:
for hash_ in pull_dicts_null:
    v = pull_dicts_null[hash_]
    print(v["binning_mode"], v["nbins"])
    print(v["pull_o0"])
    print(v["pull_o1"])
    print(v["pull_o2"])
    print(v["pull_signal"])

TypeError: list indices must be integers or slices, not str

In [11]:
print("Calibrated without correction")
display(calibrated_pairs_o0)
idx_tuples = [tuple(x) for x in calibrated_pairs_o0.values]
for t in idx_tuples:
    display(pull_df_summary_reindex.loc[t, ["signal_ratio", "rejected_o0", "rejected_o1", "rejected_o2"]])

Calibrated without correction


,ensemble_mode,bin_stats,noise_scale,nbins,binning_mode,SR_size
0,mean,smeared,0.5,256,train,0.05
1,max,smeared,2.0,256,train,0.05
2,mean,smeared,2.0,256,train,0.05
3,mean,smeared,3.0,256,train,0.05
4,max,smeared,2.0,128,train,0.10
5,max,smeared,2.0,256,train,0.15


/tmp/ipykernel_2035341/1227169898.py:5: PerformanceWarning:

indexing past lexsort depth may impact performance.



signal_ratio  \
ensemble_mode bin_stats noise_scale nbins binning_mode SR_size                 
mean          smeared   0.5         256   train        0.05           0.0000   
                                                       0.05           0.0050   
                                                       0.05           0.0075   
                                                       0.05           0.0100   
                                                       0.05           0.0200   

                                                                rejected_o0  \
ensemble_mode bin_stats noise_scale nbins binning_mode SR_size                
mean          smeared   0.5         256   train        0.05            0.04   
                                                       0.05            0.22   
                                                       0.05            0.40   
                                                       0.05            0.90   
                                                       0.05            1.00   

                                                                rejected_o1  \
ensemble_mode bin_stats noise_scale nbins binning_mode SR_size                
mean          smeared   0.5         256   train        0.05            0.02   
                                                       0.05            0.18   
                                                       0.05            0.26   
                                                       0.05            0.90   
                                                       0.05            1.00   

                                                                rejected_o2  
ensemble_mode bin_stats noise_scale nbins binning_mode SR_size               
mean          smeared   0.5         256   train        0.05            0.00  
                                                       0.05            0.10  
                                                       0.05            0.06  
                                                       0.05            0.80  
                                                       0.05            0.98

/tmp/ipykernel_2035341/1227169898.py:5: PerformanceWarning:

indexing past lexsort depth may impact performance.



...

signal_ratio  \
ensemble_mode bin_stats noise_scale nbins binning_mode SR_size                 
mean          smeared   3.0         256   train        0.05           0.0000   
                                                       0.05           0.0050   
                                                       0.05           0.0075   
                                                       0.05           0.0100   
                                                       0.05           0.0200   

                                                                rejected_o0  \
ensemble_mode bin_stats noise_scale nbins binning_mode SR_size                
mean          smeared   3.0         256   train        0.05            0.06   
                                                       0.05            0.16   
                                                       0.05            0.66   
                                                       0.05            1.00   
                                                       0.05            0.98   

                                                                rejected_o1  \
ensemble_mode bin_stats noise_scale nbins binning_mode SR_size                
mean          smeared   3.0         256   train        0.05            0.08   
                                                       0.05            0.12   
                                                       0.05            0.52   
                                                       0.05            1.00   
                                                       0.05            0.98   

                                                                rejected_o2  
ensemble_mode bin_stats noise_scale nbins binning_mode SR_size               
mean          smeared   3.0         256   train        0.05            0.08  
                                                       0.05            0.10  
                                                       0.05            0.36  
                                                       0.05            0.98  
                                                       0.05            0.98

/tmp/ipykernel_2035341/1227169898.py:5: PerformanceWarning:

indexing past lexsort depth may impact performance.



signal_ratio  \
ensemble_mode bin_stats noise_scale nbins binning_mode SR_size                 
max           smeared   2.0         128   train        0.1            0.0000   
                                                       0.1            0.0050   
                                                       0.1            0.0075   
                                                       0.1            0.0100   
                                                       0.1            0.0200   

                                                                rejected_o0  \
ensemble_mode bin_stats noise_scale nbins binning_mode SR_size                
max           smeared   2.0         128   train        0.1             0.06   
                                                       0.1             0.16   
                                                       0.1             0.50   
                                                       0.1             0.98   
                                                       0.1             1.00   

                                                                rejected_o1  \
ensemble_mode bin_stats noise_scale nbins binning_mode SR_size                
max           smeared   2.0         128   train        0.1             0.02   
                                                       0.1             0.10   
                                                       0.1             0.34   
                                                       0.1             0.98   
                                                       0.1             1.00   

                                                                rejected_o2  
ensemble_mode bin_stats noise_scale nbins binning_mode SR_size               
max           smeared   2.0         128   train        0.1             0.00  
                                                       0.1             0.08  
                                                       0.1             0.22  
                                                       0.1             0.92  
                                                       0.1             1.00

/tmp/ipykernel_2035341/1227169898.py:5: PerformanceWarning:

indexing past lexsort depth may impact performance.



signal_ratio  \
ensemble_mode bin_stats noise_scale nbins binning_mode SR_size                 
max           smeared   2.0         256   train        0.15           0.0000   
                                                       0.15           0.0050   
                                                       0.15           0.0075   
                                                       0.15           0.0100   
                                                       0.15           0.0200   

                                                                rejected_o0  \
ensemble_mode bin_stats noise_scale nbins binning_mode SR_size                
max           smeared   2.0         256   train        0.15            0.04   
                                                       0.15            0.24   
                                                       0.15            0.54   
                                                       0.15            0.98   
                                                       0.15            1.00   

                                                                rejected_o1  \
ensemble_mode bin_stats noise_scale nbins binning_mode SR_size                
max           smeared   2.0         256   train        0.15            0.02   
                                                       0.15            0.08   
                                                       0.15            0.34   
                                                       0.15            0.98   
                                                       0.15            1.00   

                                                                rejected_o2  
ensemble_mode bin_stats noise_scale nbins binning_mode SR_size               
max           smeared   2.0         256   train        0.15            0.02  
                                                       0.15            0.06  
                                                       0.15            0.20  
                                                       0.15            0.96  
                                                       0.15            1.00